# Rover on Rough Terrain — Planning Challenge

A point rover lives on a 2D arena. It must drive from **start** to **goal**.

You are given:
- `field` — a `(H, W, 3)` terrain map with 3 channels you can see:
  - **ch 0 — rocks**: hard obstacles (lethal, must avoid)
  - **ch 1 — mud**: continuous 0..1
  - **ch 2 — slope**: continuous 0..1
- `demos` — a handful of **expert trajectories** (lists of `(x, y)` points).
  Each demo runs between its *own* start/goal pair (NOT necessarily yours), so
  no single demo is the answer — they're there to reveal the terrain cost.
- `start`, `goal` — the 2D points YOUR path must connect (`x = column`, `y = row`).

**Motion model:** the rover is a point moving in continuous `(x, y)` space. Your
planner may return any polyline of waypoints; moves are not restricted to a
4-connected or 8-connected grid. For example, a grid-search implementation may
choose 8-connected neighbors and a sampling planner may produce arbitrary
diagonal segments. The metric samples along every segment, so a move cannot
skip over a rock or expensive terrain just by using sparse waypoints.

The experts know something you don't: the *true* traversal cost of the terrain.
Your job is to produce a path that reaches the goal **and** is about as
terrain-cheap as the experts'. A path that only dodges rocks will reach the
goal but cost too much — look at the demos to figure out what else to avoid.

**Metric (pre-built, do not edit):** a path passes if it
(1) starts near start, ends near goal, (2) never touches a rock,
(3) stays within the arena bounds (leaving the map fails),
(4) is traversable under your submitted cost map, and
(5) has terrain cost `<= expert_cost * tol`.

Two helpers ship with the metric cell:
- `evaluate(path, cost_map)` — the official PASS/FAIL check above.
- `path_cost(path, cost_map)` — integrates *your* cost map along a path and
  returns its cost (plus feasibility flags). Use this inside your planner to
  score candidate rollouts under the cost map you've designed.

Work through the TODOs. You can stop at the classical solution or push on to
the learned one; partial progress is fine. Talk through your choices as you go.

### Setup &mdash; imports and data load

Loads `field`, `demos`, `start`, `goal`, and the metric metadata from disk
so the rest of the notebook can use them. Re-run if you re-download the data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys

# Clone the data repo when running on Colab (no-op if files already exist).
if not os.path.exists('field.npy'):
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field.npy')                          # (H, W, 3)
demos = np.load('demos.npy', allow_pickle=True)       # object array of (T,2)
meta  = np.load('meta.npz')
start, goal = meta['start'], meta['goal']
H, W = field.shape[:2]
ROCK, MUD, SLOPE = 0, 1, 2
print('field', field.shape, '| demos', len(demos),
      '| start', start, 'goal', goal)

### Visualization helper

Defines `show(paths, cost_map, title)` &mdash; your one stop for plotting
the terrain (or any cost map), the expert demos, and one or more candidate
paths. Use it freely while debugging. The cell also calls `show()` once to
render the initial terrain.

In [ ]:
def show(paths=None, cost_map=None, title=''):
    fig, ax = plt.subplots(figsize=(6, 6))
    if cost_map is not None:
        ax.imshow(cost_map, origin='lower', cmap='viridis')
    else:
        # rocks yellow, mud red-ish, slope blue-ish
        rgb = np.zeros((H, W, 3))
        rgb[..., 0] = field[..., MUD]
        rgb[..., 2] = field[..., SLOPE]
        rgb[field[..., ROCK] > 0.5] = [1.0, 0.85, 0.1]
        ax.imshow(rgb, origin='lower')
    for d in demos:
        d = np.asarray(d)
        ax.plot(d[:, 0], d[:, 1], '0.7', lw=1, alpha=.7)
    if paths is not None:
        if isinstance(paths, np.ndarray) and paths.ndim == 2:
            paths = [paths]
        for p in paths:
            p = np.asarray(p)
            ax.plot(p[:, 0], p[:, 1], 'lime', lw=2)
    ax.plot(*start, 'wo', ms=8); ax.plot(*goal, 'w*', ms=14)
    ax.set_title(title); ax.set_xlim(0, W); ax.set_ylim(0, H)
    plt.show()

show(title='terrain (red=mud, blue=slope, yellow=rock) + expert demos (gray)')

### Pre-built metric &mdash; `path_cost` and `evaluate`

Do not edit. Two helpers are defined here:

- `path_cost(path, cost_map)` &mdash; integrates *your* cost map along a path
  and returns its cost plus feasibility flags. Use this inside your planner
  to score candidate rollouts.
- `evaluate(path, cost_map)` &mdash; the official PASS/FAIL scorer. Prints a
  one-line summary against the expert bar.

The true terrain cost lives behind these functions; recovering what drives
it from the demos is the point of the challenge, so don't peek &mdash; just
call `evaluate`.

In [ ]:
# ============================ PRE-BUILT METRIC ============================
# Two functions are provided for you:
#
#   path_cost(path, cost_map) -> dict
#       Integrates `cost_map` along `path` (after resampling to ~uniform
#       spacing, so the result doesn't depend on how many waypoints you used).
#       Returns {'cost', 'hit_high', 'out_of_bounds', 'start_ok', 'goal_ok'}.
#       USE THIS to score candidate paths under YOUR OWN cost map -- e.g.
#       inside your planner's rollout-scoring inner loop, or to compare two
#       candidate paths without invoking the PASS/FAIL bar.
#
#   evaluate(path, cost_map) -> bool
#       Official PASS/FAIL scorer. Calls path_cost twice: once with YOUR
#       cost_map (to check the path is feasible under the cost map you
#       submitted) and once with the HIDDEN true terrain cost (to score it
#       against the expert bar). Prints a one-line summary and returns
#       True iff the path passes.
#
# Do not edit. The true per-step terrain cost is shipped as an opaque grid in
# meta.npz (meta['true_cost']); recovering what drives it from the demos is
# the point of the challenge, so don't peek at it -- just call evaluate(path).
EXPERT_COST = float(meta['expert_cost'])
TOL = float(meta['tol'])
_TRUE_COST = meta['true_cost']                 # (H,W) opaque scoring grid
_ROCK = float(meta['rock_sentinel'])           # cells >= this are rock
_STEP = 0.5                                    # path-integral resampling (cells)
# Penalty applied to rock / forbidden cells. A large finite number (not inf)
# so path_cost stays numerically well-behaved -- every rollout the planner
# scores has a comparable cost even if it crosses an obstacle, which avoids
# argmin stalling on a sea of inf values.
ROCK_PENALTY = 1e6
# Two views of the hidden scoring grid:
#   _TRUE_COST_PEN  -- rocks at ROCK_PENALTY; used to detect rock crossings.
#   _TRUE_COST_BARE -- rocks zeroed out; used to report the honest terrain
#                      cost against the bar (the rock_hit flag is the signal
#                      that there was a violation; we don't double-count it
#                      into the displayed cost).
_TRUE_COST_PEN  = np.where(_TRUE_COST >= _ROCK, ROCK_PENALTY, _TRUE_COST)
_TRUE_COST_BARE = np.where(_TRUE_COST >= _ROCK, 0.0,          _TRUE_COST)

def path_cost(path, cost_map, hi_threshold=ROCK_PENALTY / 2):
    '''Integrate cost_map along path (resampled to ~uniform spacing).

    Use this to score a candidate path under YOUR cost map -- for instance to
    pick the cheapest rollout inside your planner. Returns a dict so you can
    detect untraversable-cell crossings and out-of-bounds excursions
    separately from the numerical cost.

    A "high" cell is one whose cost is >= `hi_threshold` (default 5e5, half of
    the rock penalty). Use this to mark rocks or other "do not enter" regions
    with a large finite cost rather than np.inf -- finite costs keep the
    planner's score landscape well-behaved (every rollout has a comparable
    cost, even if it crosses an obstacle).

    Returns
    -------
    dict with keys:
      cost           : float, integrated cost along the path
      hit_high       : bool, True if any resampled point landed on a cell with
                       cost >= hi_threshold (typically a rock / forbidden cell)
      out_of_bounds  : bool, True if any resampled point left the arena
      start_ok       : bool, path[0]  within 5 units of `start`
      goal_ok        : bool, path[-1] within 5 units of `goal`
    '''
    path = np.asarray(path, float)
    cost_map = np.asarray(cost_map, float)
    if cost_map.shape != (H, W):
        raise ValueError(f'cost_map must have shape {(H, W)}, got {cost_map.shape}')
    start_ok = bool(np.linalg.norm(path[0]  - start) < 5)
    goal_ok  = bool(np.linalg.norm(path[-1] - goal)  < 5)
    seg = np.linalg.norm(np.diff(path, axis=0), axis=1)
    L = float(seg.sum())
    if L < 1e-9:
        return {'cost': float('inf'), 'hit_high': False, 'out_of_bounds': False,
                'start_ok': start_ok, 'goal_ok': goal_ok}
    s = np.concatenate([[0.0], np.cumsum(seg)])
    n = max(2, int(L / _STEP) + 1)
    u = np.linspace(0, L, n)
    xs = np.interp(u, s, path[:, 0]); ys = np.interp(u, s, path[:, 1])
    hit_high = False
    out_of_bounds = False
    oob_penalty = ROCK_PENALTY     # leaving the arena costs the same as a rock
    cost = 0.0
    for x, y in zip(xs, ys):
        rx, ry = round(x), round(y)
        if not (0 <= rx <= W - 1 and 0 <= ry <= H - 1):
            out_of_bounds = True
            cost += oob_penalty * _STEP
            continue
        cell = cost_map[int(ry), int(rx)]
        if cell >= hi_threshold:
            hit_high = True
        cost += cell * _STEP
    return {'cost': cost, 'hit_high': hit_high, 'out_of_bounds': out_of_bounds,
            'start_ok': start_ok, 'goal_ok': goal_ok}

def evaluate(path, cost_map):
    '''Official PASS/FAIL scorer. Returns True iff the path passes.

    Checks (1) endpoints near start/goal, (2) the path is feasible under
    `cost_map` (no untraversable cells along it), (3) stays in bounds,
    (4) doesn't touch a rock under the hidden true cost, and (5) terrain
    cost <= expert bar. Prints a one-line summary. To score paths under your
    own cost map without invoking the bar, use path_cost instead.
    '''
    cand = path_cost(path, cost_map)
    rock_check = path_cost(path, _TRUE_COST_PEN)   # detects rock crossings
    bare = path_cost(path, _TRUE_COST_BARE)        # honest terrain-only cost
    bar = EXPERT_COST * TOL
    success = bool(cand['start_ok'] and cand['goal_ok']
                   and not cand['hit_high']
                   and not rock_check['hit_high']
                   and not rock_check['out_of_bounds']
                   and bare['cost'] <= bar)
    print(f"start_ok={cand['start_ok']} goal_ok={cand['goal_ok']} "
          f"model_feasible={not cand['hit_high']} rock_hit={rock_check['hit_high']} "
          f"out_of_bounds={rock_check['out_of_bounds']} "
          f"cost={bare['cost']:.1f} (bar={bar:.1f}) -> {'PASS' if success else 'FAIL'}")
    return success

## TODO 1 &mdash; a cost over states (start classical)

Implement `cost_map_hand(field)` to return an `(H, W)` per-cell traversal
cost. At minimum penalize rocks heavily (use `ROCK_PENALTY`, *not*
`np.inf`); optionally guess penalties for mud/slope. The placeholder is a
uniform cost of 1 that's enough to run the planner below but not enough
to pass.

In [ ]:
# ---- TODO 1: a cost over states ------------------------------------------
# Return a per-cell traversal cost map of shape (H, W). Cells the rover should
# avoid should be expensive; rocks should carry a LARGE FINITE penalty (e.g.
# 1e6 -- the metric uses ROCK_PENALTY for this) rather than np.inf. Finite
# costs keep the planner's score landscape well-behaved: every rollout has a
# comparable number, even one that crosses an obstacle, so a sampling planner
# can rank "barely-touches-a-rock" vs "way-into-a-rock" and climb out.
# A trivial uniform placeholder is provided so everything runs. Improve it:
# at minimum mark rocks as expensive; optionally guess mud/slope penalties,
# then run the planner below and see why this isn't enough.
def cost_map_hand(field):
    '''Hand-designed (H,W) cost from the (H,W,3) terrain features. No demos used.'''
    cm = np.ones((H, W), float)   # uniform placeholder -- everything costs 1
    # TODO: penalize rocks heavily (e.g. ROCK_PENALTY); optionally guess
    # mud/slope penalties.
    return cm

## TODO 2 &mdash; sampling rollout planner

Implement `plan(cost_map, ...)` to return an `(T, 2)` polyline from
`start` to `goal` that is cheap under `cost_map`. A sampling-based
rollout (keep a current trajectory, perturb it `K` ways each iteration,
keep the best under `path_cost`) is the suggested approach &mdash; see
the comments inside. The placeholder is a straight line that ignores
`cost_map` entirely.

In [ ]:
# ---- TODO 2: a sampling-based rollout planner ---------------
# Plan a path start->goal that is cheap under a given (H,W) cost_map.
# Suggested scheme (you may do something else):
#   - keep a current trajectory of T waypoints from start toward goal
#   - sample K noisy candidate rollouts around it
#   - score each by sum of cost_map along it
#     (path_cost(path, cost_map)['cost'] from the metric cell does exactly
#      this -- integrates cost_map along the path and returns the total)
#   - update toward the low-cost samples (softmax / take-best)
# Keep it on CPU/numpy first. We'll talk about batching/GPU after.
def plan(cost_map, K=256, T=60, iters=30, seed=0):
    '''Return an (T,2) path of (x,y) points from start to goal.'''
    rng = np.random.default_rng(seed)
    # Trivial placeholder: a straight line start->goal. It ignores cost_map
    # entirely (so it drives through whatever is in the way). Replace with a
    # sampling rollout that actually uses cost_map.
    # TODO: implement the sampling rollout loop.
    return np.linspace(start, goal, T)

### Run the classical (hand-cost) solution

Plans a path under your `cost_map_hand`, visualizes it, and scores it.
With only rocks made expensive, this should *reach the goal but cost too
much* &mdash; you'll see `FAIL` and that's the expected motivation for
TODO 3.

In [ ]:
# Run the hand-cost classical solution:
path = plan(cost_map_hand(field))
show(path, title='classical: hand cost')
evaluate(path, cost_map_hand(field))   # expect: reaches goal, but terrain cost over the bar

## TODO 3 &mdash; learn the cost from demos

Implement `cost_map_learned(field, demos)` to infer a cost map from the
expert demonstrations. The experts detour around something the rock map
alone doesn't capture; figure out what they avoid and bake that into the
cost. Several approaches are sketched in the comments &mdash; pick one
and justify it. The placeholder just falls back to `cost_map_hand`.

In [ ]:
# ---- TODO 3: learn the cost from the expert demos ------------------------
# The experts detour around something the rock map doesn't show. Use the demos
# to build a cost map that explains their behavior, then re-plan.
# Many valid approaches — pick one and justify it:
#   - visitation: cells experts traverse are cheap, cells they avoid costly
#   - feature regression / IRL: fit weights w over [mud, slope] so expert
#     paths are low-cost relative to alternatives
#   - nearest-demo / warping
def cost_map_learned(field, demos):
    '''Return an (H,W) cost map inferred from demos. Keep rocks impassable.'''
    # Placeholder: same uniform cost as the hand version (ignores demos).
    # Replace this with something that USES the demos.
    # TODO: your approach here
    return cost_map_hand(field)

### Run the learned solution

Builds your learned cost map, visualizes it, plans under it, and scores
against the bar. Goal: PASS.

In [ ]:
# Run the learned solution:
cm = cost_map_learned(field, demos)
show(cost_map=cm, title='learned cost map')
path = plan(cm)
show(path, title='learned: planned path')
evaluate(path, cm)   # goal: PASS

### Bonus / discussion — make it fast

Your planner rolls out `K` trajectories every iteration. On a real robot
we replan in a closed loop at, say, 50 Hz with `K` in the thousands.

- **Vectorize** the rollout: evaluate all `K` candidates without a Python loop
  (batched numpy, or move the cost lookup + scoring to `torch` on GPU).
- Be ready to discuss: memory layout of the `K × T × 2` buffer, how `K` trades
  off against horizon `T`, host↔device transfer / sync cost, and what your
  per-replan time budget buys you at 50 Hz.

In [ ]:
# ---- BONUS TODO: vectorized / GPU rollout --------------------------------
# Re-implement the per-iteration rollout scoring without a Python loop over K.
# numpy: index cost_map with a (K, T) array of flattened cell ids.
# torch: keep cost_map + samples on cuda; one gather + sum.
# (Optional — sketch it even if you don't finish.)